# Observabilidade e reprodução de sessão do navegador 

## Visão Geral

Neste tutorial, aprenderemos como adicionar observabilidade a uma sessão do Agentcore Browser e visualizar logs do console do navegador, logs de rede, comandos CDP enviados e ações do agente executadas no navegador. 


### Detalhes do Tutorial


| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                   |
| Tipo de agente      | Único                                                                            |
| Framework Agêntico  | Nova Act                                                                         |
| Modelo LLM          | Modelo Amazon Nova Act                                                           |
| Componentes         | Observar logs do navegador no console do Agentcore Browser                       |
| Vertical            | vertical                                                                         |
| Complexidade        | Fácil                                                                            |
| SDK utilizado       | Amazon Bedrock AgentCore Python SDK, boto3 SDK, Nova Act                         |

### Arquitetura do Tutorial

Neste tutorial, veremos como observar logs do console do navegador, logs de rede, comandos CDP e ações do agente executadas no navegador.   


### Principais Recursos do Tutorial

* Habilitar gravação de sessão para a ferramenta de navegador 
* Usar Nova Act com a ferramenta de navegador
* Observar os logs e a reprodução de sessão

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Amazon Boto3 SDK
* Nova Act SDK e chave de API - Navegue até https://nova.amazon.com/act para gerar uma chave de API

In [ ]:
!pip install  -r requirements.txt --quiet

## Criar um recurso personalizado do AgentCore Browser com gravação habilitada
Primeiro, precisamos criar um recurso de ferramenta de navegador com gravação habilitada, pois a ferramenta de navegador padrão não tem gravação ativada. Em seguida, iniciaremos uma sessão de navegador usando este recurso de navegador. 

In [ ]:
## Criar um bucket S3 para armazenar gravações do navegador
## Se você quiser usar um bucket existente, pule esta etapa e atualize o nome do bucket na próxima etapa.
import boto3
import uuid
from boto3.session import Session

boto_session = Session()

region = boto_session.region_name
s3_client = boto3.client('s3', region_name=region)

bucket_name = f"agentcore-browser-recordings-{str(uuid.uuid4())[:8]}"
s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
print(f"Bucket S3 criado: {bucket_name}")

### Criar a role de execução com permissões necessárias para criar o recurso de ferramenta de navegador

Vamos criar uma função auxiliar para criar a role de execução com as permissões corretas. 

In [ ]:
## Criar role de execução com permissões para criar navegador 
def create_agentcore_role(agent_name):
    iam_client = boto3.client('iam')
    agentcore_role_name = f'agentcore-{agent_name}-role'
    boto_session = Session()
    region = boto_session.region_name
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    
    role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BrowserPermissions",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:ConnectBrowserAutomationStream",
                    "bedrock-agentcore:ListBrowsers",
                    "bedrock-agentcore:GetBrowserSession",
                    "bedrock-agentcore:ListBrowserSessions",
                    "bedrock-agentcore:CreateBrowser",
                    "bedrock-agentcore:StartBrowserSession",
                    "bedrock-agentcore:StopBrowserSession",
                    "bedrock-agentcore:ConnectBrowserLiveViewStream",
                    "bedrock-agentcore:UpdateBrowserStream",
                    "bedrock-agentcore:DeleteBrowser",
                    "bedrock-agentcore:GetBrowser"
                ],
                "Resource": "*"
            },
            {
                "Sid": "S3Permissions",
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:GetObject",
                    "s3:ListBucket"
                ],
                "Resource": [
                    f"arn:aws:s3:::{bucket_name}",
                    f"arn:aws:s3:::{bucket_name}/*"
                ]
            },
            {
                "Sid": "CloudWatchLogsPermissions",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                    "logs:DescribeLogStreams"
                ],
                "Resource": "*"
            }
        ]
    }
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock-agentcore.amazonaws.com"
                },
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {
                        "aws:SourceAccount": f"{account_id}"
                    },
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:*"
                    }
                }
            }
        ]
    }

    assume_role_policy_document_json = json.dumps(
        assume_role_policy_document
    )
    role_policy_document = json.dumps(role_policy)

    try:
        # Criar IAM Role com a trust policy
        agentcore_iam_role = iam_client.create_role(
            RoleName=agentcore_role_name,
            AssumeRolePolicyDocument=assume_role_policy_document_json,
        )
        print(f"Role {agentcore_role_name} criada com sucesso.")

        # Anexar a política de permissões inline à role
        iam_client.put_role_policy(
            RoleName=agentcore_role_name,
            PolicyName=f'{agentcore_role_name}-inline-policy',
            PolicyDocument=role_policy_document
        )
        print(f"Política inline anexada à role {agentcore_role_name}.")

    except iam_client.exceptions.EntityAlreadyExistsException:
        print("Role já existe -- deletando e criando novamente")
        
        # Desanexar e deletar políticas inline existentes
        policies = iam_client.list_role_policies(RoleName=agentcore_role_name)
        for policy_name in policies['PolicyNames']:
            iam_client.delete_role_policy(
                RoleName=agentcore_role_name,
                PolicyName=policy_name
            )
        
        # Deletar e recriar a role
        print(f"Deletando role {agentcore_role_name}...")
        iam_client.delete_role(RoleName=agentcore_role_name)
        print(f"Recriando role {agentcore_role_name}...")
        
        agentcore_iam_role = iam_client.create_role(
            RoleName=agentcore_role_name,
            AssumeRolePolicyDocument=assume_role_policy_document_json
        )
        print(f"Role {agentcore_role_name} recriada com sucesso.")

        # Reanexar a política de permissões inline à role recriada
        iam_client.put_role_policy(
            RoleName=agentcore_role_name,
            PolicyName=f'{agentcore_role_name}-inline-policy',
            PolicyDocument=role_policy_document
        )
        print(f"Política inline reanexada à role {agentcore_role_name}.")

    # Pausa para garantir que as alterações sejam propagadas
    time.sleep(10)
    
    return agentcore_iam_role

### Criar o recurso de ferramenta de navegador com gravação habilitada

In [ ]:
## Usar boto3 para criar um recurso de ferramenta de navegador personalizado com gravação habilitada
import boto3
import time
import json
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

cp_client = boto3.client('bedrock-agentcore-control', region_name=region)

# Definir parâmetros para a ferramenta de navegador
browser_name = "my_custom_browser"
browser_description = "Navegador de teste para observabilidade e reprodução de sessão"
execution_role = create_agentcore_role(browser_name)
execution_role_arn = execution_role['Role']['Arn'] # Substituir pelo ARN da sua IAM role
s3_bucket_name = bucket_name # Substituir pelo nome do seu bucket S3 se você tiver um bucket existente
s3_prefix = "replay-data"

try:
    response = cp_client.create_browser(
        name=browser_name,
        description=browser_description,
        networkConfiguration={
            "networkMode": "PUBLIC" # Ou "VPC" se você precisar de integração VPC
        },
        executionRoleArn=execution_role_arn,
        clientToken=str(uuid.uuid4()), # Token único para idempotência
        recording={
            "enabled": True,
            "s3Location": {
                "bucket": s3_bucket_name,
                "prefix": s3_prefix
            }
        }
    )
    print(response)
    print(f"Ferramenta de navegador criada com sucesso: {response['browserId']}")
    browserId = response['browserId']
except cp_client.exceptions.ConflictException as e:
    print("Ferramenta de navegador com este nome já existe. Por favor, escolha um nome diferente.")


## Criar o script do Nova Act
O Nova Act iniciará uma sessão de navegador usando o recurso de ferramenta de navegador que criamos na etapa anterior e executará as ações do navegador nele.

In [ ]:
%%writefile basic_browser_with_nova_act.py
"""Script de automação do navegador usando Amazon Bedrock AgentCore e Nova Act.

Este script demonstra automação web com IA ao:
- Inicializar uma sessão de navegador através do Amazon Bedrock AgentCore
- Conectar ao Nova Act para interações web em linguagem natural
- Realizar buscas automatizadas e extração de dados usando o navegador
"""

from bedrock_agentcore.tools.browser_client import browser_session , BrowserClient
from nova_act import NovaAct
from rich.console import Console
import argparse
import json

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("usando região", region)

def browser_with_nova_act(prompt, starting_page, nova_act_key,  browserId, region="us-west-2"):
    result = None
    
    browser_client = BrowserClient(region)
    browser_client.start(identifier=browserId) # Usar o ID da ferramenta de navegador criado aqui
    
    ws_url, headers = browser_client.generate_ws_headers()
    try:
        with NovaAct(
            cdp_endpoint_url=ws_url,
            cdp_headers=headers,
            nova_act_api_key=nova_act_key,
            starting_page=starting_page,
        ) as nova_act:
            result = nova_act.act(prompt)
    except Exception as e:
        console.print(f"Erro do NovaAct: {e}")

    finally:
        browser_client.stop()
        return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Instrução de busca no navegador")
    parser.add_argument("--starting-page", required=True, help="URL inicial")
    parser.add_argument("--nova-act-key", required=True, help="Chave de API do Nova Act")
    parser.add_argument("--region", default="us-west-2", help="Região AWS")
    parser.add_argument("--browserID", required=True, help="ID da ferramenta de navegador a usar")
    args = parser.parse_args()

    result = browser_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.browserID, args.region
    )
    console.print(f"\n[cyan] Resposta[/cyan] {result.response}")
    console.print(f"\n[bold green]Resultado do Nova Act:[/bold green] {result}")

#### Executando o script
Cole sua chave de API do Nova Act abaixo antes de executar o script. 

In [ ]:
NOVA_ACT_KEY= '' ### Cole sua chave do Nova Act aqui

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Pesquise por macbooks e extraia os detalhes do primeiro" --starting-page "https://www.amazon.com/" --browserID {browserId} --nova-act-key {NOVA_ACT_KEY}

## Observabilidade no Console do Agentcore Browser
* Enquanto o script estiver em execução, você pode acessar o console AWS: https://us-west-2.console.aws.amazon.com/bedrock-agentcore/builtInTools
e clicar na aba "Browser use tools". Se você estiver executando em uma região diferente - substitua a região neste URL.
* Clique em "my-custom-browser". Você verá um link para visualizar a visualização ao vivo se a sessão ainda estiver em execução ou um link para visualizar a gravação. 
* Aguarde a sessão terminar se estiver em execução e então clique em visualizar gravação. 
Você verá uma página similar a esta

![image](./images/browser_recording_1.png)

* #### Agora você pode reproduzir a sessão gravada do navegador 
* #### Inspecionar cada página visitada durante a sessão 
* #### Visualizar as ações executadas pelo agente na aba Action 
* #### Visualizar os detalhes do DOM da página, logs do Console, comandos CDP enviados ao navegador e os logs de Rede 
* #### Você pode baixar cada um dos logs para depuração adicional
* #### Você pode clicar em "View" na aba Actions para cada ação para visualizar a ação exata executada no navegador 
* #### Por exemplo: Visualize qualquer ação do tipo "Click" e observe o círculo vermelho no navegador. Isso indica a localização exata onde a ação de clique ocorreu. 

# Parabéns e boa exploração! 